# Check a quotation (citations)

A paper quotes something. `citations` reads the source and reports whether the passage is
there — and whether the file it read is the file the record pinned.

Every cell runs the **published package**.

## Install

In [ ]:
import piplite

await piplite.install(["citations==0.2.0", "pypdf"])
print("installed")

In [ ]:
import sys


def cli(module, *args):
    old = sys.argv
    sys.argv = [module.split(".")[0], *args]
    try:
        __import__(module, fromlist=["main"]).main()
    except SystemExit:
        pass
    finally:
        sys.argv = old

## A source, and a claim about it

In [ ]:
import os
import pathlib

os.makedirs("/tmp/cite", exist_ok=True)
os.chdir("/tmp/cite")

pathlib.Path("source.txt").write_text(
    "We found that the treatment reduced the primary outcome (p = 0.031, n = 200). "
    "Adverse events were monitored for 30 days."
)

pathlib.Path("claims").mkdir(exist_ok=True)
pathlib.Path("claims/effect.yaml").write_text("""source:
  citation: Someone et al. 2026
  local: source.txt
evidence:
  primary:
    quotes:
      - exact: "the treatment reduced the primary outcome"
      - exact: "monitored for 30 days"
""")
cli("citations.cli", "verify", "--claims", "claims")

## A quotation that is not there

The tool reads the source and says so. It never reports a passage as present because it
looks similar to one.

In [ ]:
pathlib.Path("claims/effect.yaml").write_text("""source:
  citation: Someone et al. 2026
  local: source.txt
evidence:
  primary:
    quotes:
      - exact: "the treatment increased the primary outcome"
""")
cli("citations.cli", "verify", "--claims", "claims")

## The source changed after it was pinned

A record can pin a source by digest. Every quotation can still resolve, against a document
that is not the one the record describes.

In [ ]:
import hashlib

digest = hashlib.sha256(pathlib.Path("source.txt").read_bytes()).hexdigest()

pathlib.Path("claims/effect.yaml").write_text(f"""source:
  citation: Someone et al. 2026
  local: source.txt
  sha256: {digest}
evidence:
  primary:
    quotes:
      - exact: "the treatment reduced the primary outcome"
""")
print("--- pinned, unchanged ---")
cli("citations.cli", "verify", "--claims", "claims")

text = pathlib.Path("source.txt").read_text()
pathlib.Path("source.txt").write_text(text.replace("p = 0.031", "p = 0.043"))
print("\n--- after editing a number the quotation does not cite ---")
cli("citations.cli", "verify", "--claims", "claims")

---
## PDFs

The published package reads a PDF by calling `pdftotext`, which is a binary and cannot run
in WebAssembly. The extractor is a single function, so this notebook substitutes a
pure-Python one and the rest of the package is unchanged.

On your own machine, `pdftotext` from poppler is used and no substitution is needed.

In [ ]:
import citations.verify as V
import pypdf


def extract(path, page=None):
    """Read a source. Text files are read directly, exactly as the package does;
    only the PDF branch differs, and only because pdftotext is a binary."""
    path = pathlib.Path(path)
    if path.suffix.lower() in V.TEXT_SUFFIXES:
        return path.read_text(errors="replace")
    reader = pypdf.PdfReader(str(path))
    pages = reader.pages if page is None else [reader.pages[page - 1]]
    return "\n".join(p.extract_text() or "" for p in pages)


extract.cache_clear = lambda: None
V.extract = extract
print(extract(pathlib.Path("/drive/paper.pdf"))[:110])

### A quotation in the PDF

In [ ]:
# `/drive` is Emscripten's view of the notebook's bundled files. Copy the bytes
# explicitly: shutil.copy across that boundary does not preserve them.
pathlib.Path("paper.pdf").write_bytes(pathlib.Path("/drive/paper.pdf").read_bytes())
print("paper.pdf:", pathlib.Path("paper.pdf").read_bytes()[:5])

pathlib.Path("claims/pdf.yaml").write_text("""source:
  citation: Someone et al. 2026, the PDF
  local: paper.pdf
evidence:
  primary:
    quotes:
      - exact: "Adverse events were monitored for 30 days"
""")
cli("citations.cli", "verify", "--claims", "claims")

### The right passage, the wrong page

`Table 2 reports n = 147` is on page 2. Claiming it appears on page 1 is a different
failure from claiming it appears at all, and gets a different answer.

In [ ]:
pathlib.Path("claims/pdf.yaml").write_text("""source:
  citation: Someone et al. 2026, the PDF
  local: paper.pdf
evidence:
  primary:
    quotes:
      - exact: "Table 2 reports n = 147"
        page: 1
""")
cli("citations.cli", "verify", "--claims", "claims")

---
## Not possible in WebAssembly

`prereg freeze` records the commit a plan was frozen at. Git does not exist in
WebAssembly, so the command cannot run here.

```bash
pip install reproducible-science
```